# Week 11 — Notebook 2: K-Means Clustering

**Difficulty:** ⭐⭐⭐ (Intermediate)  
**Estimated Time:** 60 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. **Implement** Lloyd's K-means algorithm from scratch in pure NumPy.
2. **Implement** K-means++ initialisation from scratch and explain why it outperforms random initialisation.
3. **Explain** the convergence guarantee and time complexity O(n · k · d · i).
4. **Verify** your scratch implementation against sklearn's `KMeans` using a contingency matrix.
5. **Apply** K-means to the synthetic retail dataset and interpret the resulting customer segments.


## Real-World Analogy: Sorting Laundry

**K-means is like sorting laundry into k piles by repeatedly re-centring each pile.**

Here is the process step by step:

1. **Start:** Drop k random markers on the floor (your initial centroids).
2. **Assign:** Each sock goes to the nearest marker — it joins that pile.
3. **Update:** Move each marker to the *average position* of all socks in its pile — the new centre of that pile.
4. **Repeat steps 2–3** until no sock changes piles — convergence.

The algorithm is guaranteed to stop because there are only finitely many ways to assign n socks to k piles, and the total "travel distance" (inertia) never increases after an update step.

The tricky part: where you drop the initial markers matters a lot. Drop them all in one corner and you get a bad partition. This is the problem K-means++ solves.


## Why Does This Matter?

K-means is one of the most widely deployed algorithms in all of machine learning:

| Application | How K-means is used |
|---|---|
| **Customer segmentation** | Group customers by RFM features → targeted marketing |
| **Image compression** | Quantise pixel colours to k representative colours (vector quantisation) |
| **Document clustering** | Group news articles, support tickets, or research papers by topic |
| **Initialising other algorithms** | K-means centroids are used to initialise GMMs and some neural networks |
| **Feature engineering** | Replace raw features with cluster assignments (cluster distance features) |
| **Hardware / edge ML** | Extremely fast at inference — just compute k distances |

Understanding K-means from scratch also gives you the vocabulary to understand its many descendants: K-medoids, fuzzy K-means, mini-batch K-means, and spectral clustering.


In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, confusion_matrix
from scipy.optimize import linear_sum_assignment   # for label-permutation matching
from matplotlib.animation import FuncAnimation
from matplotlib.lines import Line2D                # for custom legend entries

# Reproducibility
np.random.seed(42)

# Consistent plot style across the whole notebook
sns.set_theme(style='whitegrid')

print("Imports successful.")


## Lloyd's Algorithm — Plain-English Pseudocode

Lloyd's algorithm (1957, published 1982) is the standard K-means procedure. It has exactly four steps:

---

**Step 1 — Initialise:** Pick k starting centroids. (Random choice for now; K-means++ later.)

**Step 2 — Assignment step (E-step):**  
For each data point x_i, compute its Euclidean distance to every centroid μ_j.  
Assign x_i to the cluster whose centroid is nearest:  
> label(x_i) = argmin_j  ‖x_i − μ_j‖²

**Step 3 — Update step (M-step):**  
Move each centroid to the mean of all points currently assigned to it:  
> μ_j ← mean({ x_i : label(x_i) = j })

**Step 4 — Convergence check:**  
If the centroids moved less than a tolerance ε (or maximum iterations reached), stop.  
Otherwise go back to Step 2.

---

**Objective being minimised (inertia / WCSS):**

> J = Σ_i  min_j  ‖x_i − μ_j‖²

In plain English: the total squared distance from each point to its nearest centroid. K-means finds a **local minimum** of J — not guaranteed to be global.


In [ ]:
np.random.seed(42)

def kmeans_scratch(X, k, max_iter=300, tol=1e-4, random_state=42):
    """
    Lloyd's K-means algorithm implemented from scratch.

    Parameters
    ----------
    X            : array of shape (n_samples, n_features)
    k            : number of clusters
    max_iter     : maximum number of iterations before forced stop
    tol          : convergence threshold on centroid shift (L2 norm)
    random_state : integer seed for the random number generator

    Returns
    -------
    labels    : array of shape (n_samples,)  — cluster index for each point
    centroids : array of shape (k, n_features) — final centroid positions
    n_iter    : number of iterations until convergence
    inertia   : final within-cluster sum of squared distances
    """
    rng = np.random.default_rng(random_state)   # independent RNG to avoid global state

    # ── Step 1: Random initialisation ─────────────────────────────────────────
    # Pick k data points at random (without replacement) as starting centroids
    idx = rng.choice(len(X), size=k, replace=False)
    centroids = X[idx].copy()                   # shape: (k, n_features)

    labels = np.zeros(len(X), dtype=int)        # will be filled in the loop

    for iteration in range(max_iter):

        # ── Step 2: Assignment step ────────────────────────────────────────────
        # Compute distance from every point to every centroid.
        # X[:, None] has shape (n, 1, d); centroids[None, :] has shape (1, k, d)
        # Broadcasting gives dists of shape (n, k)
        dists = np.linalg.norm(
            X[:, None] - centroids[None, :],    # difference tensor (n, k, d)
            axis=2                              # collapse feature dimension
        )                                       # result shape: (n, k)

        # Each point gets the index of the nearest centroid
        labels = np.argmin(dists, axis=1)       # shape: (n,)

        # ── Step 3: Update step ────────────────────────────────────────────────
        # New centroid = mean of all points assigned to that cluster
        new_centroids = np.array([
            X[labels == j].mean(axis=0)         # mean over all points in cluster j
            if np.any(labels == j)              # guard against empty clusters
            else centroids[j]                   # keep old centroid if cluster is empty
            for j in range(k)
        ])                                      # shape: (k, d)

        # ── Step 4: Convergence check ──────────────────────────────────────────
        # Measure how far all centroids moved in total
        shift = np.linalg.norm(new_centroids - centroids)  # scalar
        centroids = new_centroids

        if shift < tol:                         # centroids barely moved: converged
            break

    # ── Compute final inertia (WCSS) ───────────────────────────────────────────
    # For each point, squared distance to its assigned centroid
    # centroids[labels] selects the centroid for each point — shape (n, d)
    inertia = np.sum((X - centroids[labels]) ** 2)

    return labels, centroids, iteration + 1, inertia


# ── Quick smoke test on a simple dataset ─────────────────────────────────────
X_test, y_test = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
labels_s, centroids_s, n_iter_s, inertia_s = kmeans_scratch(X_test, k=4)

print(f"Converged after {n_iter_s} iterations")
print(f"Final inertia (WCSS): {inertia_s:.2f}")
print(f"Labels shape: {labels_s.shape}, unique labels: {np.unique(labels_s)}")


In [ ]:
np.random.seed(42)

# ── Visualise 4 snapshots of Lloyd's algorithm converging ────────────────────
# We re-run Lloyd's step by step, saving centroids and labels at specific iterations.

def kmeans_steps(X, k, n_snapshots=(0, 1, 3, 'final'), random_state=0):
    """Run K-means and return (labels, centroids) at requested iterations."""
    rng = np.random.default_rng(random_state)
    idx = rng.choice(len(X), size=k, replace=False)
    centroids = X[idx].copy()

    snapshots = {}                              # dict: iteration_number -> (labels, centroids)
    # Capture the initial centroid positions before any assignment
    snapshots['init'] = (np.full(len(X), -1), centroids.copy())

    for iteration in range(50):                 # 50 iterations is plenty for blobs
        dists = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
        labels = np.argmin(dists, axis=1)

        # Save snapshots at requested iterations
        key = iteration + 1                     # label iterations starting from 1
        snapshots[key] = (labels.copy(), centroids.copy())

        new_centroids = np.array([
            X[labels == j].mean(axis=0) if np.any(labels == j) else centroids[j]
            for j in range(k)
        ])
        shift = np.linalg.norm(new_centroids - centroids)
        centroids = new_centroids
        if shift < 1e-6:
            break

    snapshots['final'] = (labels.copy(), centroids.copy())
    return snapshots


X_vis, _ = make_blobs(n_samples=250, centers=4, cluster_std=0.9, random_state=7)
snaps = kmeans_steps(X_vis, k=4, random_state=0)

snap_keys = ['init', 1, 3, 'final']            # which iterations to display
snap_titles = [
    'Step 1: Random Initialisation\n(no assignments yet)',
    'Step 2: After Iteration 1\n(first assignment + update)',
    'Step 3: After Iteration 3',
    'Step 4: Converged'
]

palette = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, key, title in zip(axes, snap_keys, snap_titles):
    labels_snap, cents_snap = snaps[key]

    if key == 'init':
        # No assignments yet — show all points in grey
        ax.scatter(X_vis[:, 0], X_vis[:, 1],
                   c='lightgrey', s=25, alpha=0.6, edgecolors='white', linewidths=0.3)
    else:
        # Colour each point by its current cluster assignment
        for cid in range(4):
            mask = labels_snap == cid
            ax.scatter(X_vis[mask, 0], X_vis[mask, 1],
                       c=palette[cid], s=25, alpha=0.6,
                       edgecolors='white', linewidths=0.3)

    # Always show current centroid positions as black X markers
    ax.scatter(cents_snap[:, 0], cents_snap[:, 1],
               marker='X', s=200, c='black', zorder=10,
               edgecolors='white', linewidths=0.8)

    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

fig.suptitle("Lloyd's Algorithm: Snapshots of Convergence (X = centroid)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## K-Means++ Initialisation — Why Random Init Can Fail

### The problem with random initialisation

If you pick starting centroids uniformly at random, you might accidentally choose two centroids that are both inside the same cluster. That cluster then gets split between two "territories" and another cluster is left entirely uncovered.

**Adversarial example:** Imagine 3 clusters arranged in a triangle. If random init places two centroids inside the top cluster, the third centroid must serve both bottom clusters. The algorithm will converge, but to a bad local minimum — higher inertia than necessary.

This means you need many restarts with random init to find a good solution.

### The D² sampling idea (K-means++)

Arthur & Vassilvitskii (2007) proposed a smarter initialisation:

1. **Pick the first centroid** uniformly at random from the data.
2. **For each remaining centroid:**  
   a. Compute D(x) = distance from each point x to the *nearest already-chosen centroid*.  
   b. Sample the next centroid with probability proportional to **D(x)²**.  
   → Points far from all existing centroids are more likely to be chosen.
3. Repeat until you have k centroids, then run standard Lloyd's.

**Result:** K-means++ achieves an expected inertia at most O(log k) times the optimal — a provable approximation guarantee. In practice, it typically needs far fewer restarts.


In [ ]:
np.random.seed(42)

def kmeans_plus_plus_init(X, k, random_state=42):
    """
    K-means++ initialisation: choose k starting centroids using D^2 sampling.

    Parameters
    ----------
    X            : array of shape (n_samples, n_features)
    k            : number of centroids to choose
    random_state : integer seed

    Returns
    -------
    centroids : array of shape (k, n_features)
    """
    rng = np.random.default_rng(random_state)

    # Step 1: choose the first centroid uniformly at random
    first_idx = rng.integers(len(X))           # random integer in [0, n)
    centroids = [X[first_idx].copy()]          # list of chosen centroids so far

    for _ in range(k - 1):                     # we need k-1 more centroids

        # For each data point, compute its squared distance to the nearest
        # centroid chosen so far — this is D(x)^2
        d_sq_per_centroid = [
            np.linalg.norm(X - c, axis=1) ** 2   # squared dist to centroid c, shape (n,)
            for c in centroids
        ]                                        # list of k arrays, each shape (n,)

        # For each point, keep only the MINIMUM squared distance across all centroids
        D_sq = np.min(d_sq_per_centroid, axis=0)  # shape (n,)

        # Normalise to get a probability distribution over data points
        # Points far from all existing centroids get higher probability
        probs = D_sq / D_sq.sum()              # shape (n,), sums to 1

        # Sample the next centroid index according to D^2 probabilities
        next_idx = rng.choice(len(X), p=probs)
        centroids.append(X[next_idx].copy())   # add chosen point as centroid

    return np.array(centroids)                 # shape (k, d)


# Smoke test: verify we get k centroids, each a real data point
X_test2, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
init_cents = kmeans_plus_plus_init(X_test2, k=4, random_state=42)
print("K-means++ initial centroids shape:", init_cents.shape)
print("Centroids:")
print(np.round(init_cents, 3))


In [ ]:
np.random.seed(42)

# ── Compare random init vs. K-means++ init over 20 runs each ─────────────────
# For each run we record the final inertia after running Lloyd's to convergence.

X_cmp, _ = make_blobs(n_samples=500, centers=5, cluster_std=1.2, random_state=99)
k_cmp = 5
n_runs = 20

inertias_random = []
inertias_plusplus = []

for seed in range(n_runs):
    # Random init: use our kmeans_scratch with random initialisation
    _, _, _, inertia_r = kmeans_scratch(X_cmp, k=k_cmp, random_state=seed)
    inertias_random.append(inertia_r)

    # K-means++ init: initialise with ++ then run Lloyd's from those centroids
    rng_pp = np.random.default_rng(seed)
    init_c = kmeans_plus_plus_init(X_cmp, k=k_cmp, random_state=seed)
    # Run assignment/update loop starting from the ++ centroids
    centroids_pp = init_c.copy()
    for _ in range(300):
        dists_pp = np.linalg.norm(X_cmp[:, None] - centroids_pp[None, :], axis=2)
        labs_pp = np.argmin(dists_pp, axis=1)
        new_c = np.array([
            X_cmp[labs_pp == j].mean(axis=0) if np.any(labs_pp == j) else centroids_pp[j]
            for j in range(k_cmp)
        ])
        if np.linalg.norm(new_c - centroids_pp) < 1e-4:
            break
        centroids_pp = new_c
    inertia_pp = np.sum((X_cmp - centroids_pp[labs_pp]) ** 2)
    inertias_plusplus.append(inertia_pp)

# ── Side-by-side distribution plots ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

axes[0].hist(inertias_random, bins=8, color='#e41a1c', alpha=0.7, edgecolor='white')
axes[0].axvline(np.mean(inertias_random), color='darkred',
                linestyle='--', lw=2, label=f'Mean={np.mean(inertias_random):.0f}')
axes[0].set_title("Random Init\n(20 runs)", fontsize=12)
axes[0].set_xlabel("Final Inertia (lower = better)")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(inertias_plusplus, bins=8, color='#377eb8', alpha=0.7, edgecolor='white')
axes[1].axvline(np.mean(inertias_plusplus), color='darkblue',
                linestyle='--', lw=2, label=f'Mean={np.mean(inertias_plusplus):.0f}')
axes[1].set_title("K-means++ Init\n(20 runs)", fontsize=12)
axes[1].set_xlabel("Final Inertia (lower = better)")
axes[1].legend()

fig.suptitle("Random Init vs. K-means++: Inertia Distribution Over 20 Seeds",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Random init  — Mean inertia: {np.mean(inertias_random):.1f}, Std: {np.std(inertias_random):.1f}")
print(f"K-means++ init — Mean inertia: {np.mean(inertias_plusplus):.1f}, Std: {np.std(inertias_plusplus):.1f}")
print()
print("Observation: K-means++ consistently achieves lower inertia with less variance.")


## Time Complexity: O(n · k · d · i)

Let's break down where the computation goes in each iteration of Lloyd's algorithm:

| Factor | Symbol | Meaning | Typical value |
|---|---|---|---|
| **Samples** | n | Number of data points | 10,000 – 10,000,000 |
| **Clusters** | k | Number of clusters chosen | 2 – 100 |
| **Dimensions** | d | Number of features | 2 – 10,000 |
| **Iterations** | i | Iterations to convergence | 10 – 300 |

**Why O(n · k · d) per iteration:**  
The assignment step computes the distance from each of the n points to each of the k centroids. Each distance computation takes O(d) time (sum over d features). Total: n × k × d.

**Practical implications:**
- K-means scales **linearly in n** — doubling the data doubles the runtime. This is good!
- K-means scales **linearly in d** — it handles high dimensions reasonably (unlike exact nearest-neighbour search).
- The bottleneck in practice is large n (millions of points). Solutions: Mini-Batch K-Means (NB03).
- K-means++ init adds O(n · k · d) overhead — one pass per additional centroid. Usually negligible.


In [ ]:
np.random.seed(42)

# ── Full KMeansScratch class — combines Lloyd's + K-means++ init ──────────────

class KMeansScratch:
    """
    K-Means clustering from scratch.
    Supports both random and K-means++ initialisation.
    API mirrors sklearn's KMeans for easy comparison.
    """

    def __init__(self, k=3, init='k-means++', max_iter=300, tol=1e-4, random_state=42):
        """
        Parameters
        ----------
        k            : number of clusters
        init         : 'k-means++' or 'random'
        max_iter     : maximum number of Lloyd's iterations
        tol          : convergence tolerance on centroid shift
        random_state : seed for reproducibility
        """
        self.k = k
        self.init = init
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

        # Attributes set after fitting
        self.cluster_centers_ = None   # shape (k, d)
        self.labels_ = None            # shape (n,)
        self.inertia_ = None           # scalar
        self.n_iter_ = None            # scalar

    def _init_centroids(self, X):
        """Choose k initial centroids using the specified strategy."""
        rng = np.random.default_rng(self.random_state)

        if self.init == 'random':
            # Uniform random sample from the data
            idx = rng.choice(len(X), size=self.k, replace=False)
            return X[idx].copy()

        elif self.init == 'k-means++':
            # D^2 weighted sampling (see kmeans_plus_plus_init above)
            return kmeans_plus_plus_init(X, k=self.k,
                                         random_state=self.random_state)
        else:
            raise ValueError(f"Unknown init strategy: {self.init}. Use 'random' or 'k-means++'.")

    def fit(self, X):
        """Fit the model to X. Sets cluster_centers_, labels_, inertia_, n_iter_."""
        X = np.asarray(X, dtype=float)          # ensure float array
        centroids = self._init_centroids(X)     # shape (k, d)
        labels = np.zeros(len(X), dtype=int)

        for iteration in range(self.max_iter):
            # Assignment step
            dists = np.linalg.norm(
                X[:, None] - centroids[None, :], axis=2
            )                                   # shape (n, k)
            labels = np.argmin(dists, axis=1)   # shape (n,)

            # Update step
            new_centroids = np.array([
                X[labels == j].mean(axis=0)
                if np.any(labels == j) else centroids[j]
                for j in range(self.k)
            ])                                  # shape (k, d)

            # Convergence check
            shift = np.linalg.norm(new_centroids - centroids)
            centroids = new_centroids
            if shift < self.tol:
                break

        # Store fitted attributes (sklearn naming convention)
        self.cluster_centers_ = centroids
        self.labels_ = labels
        self.inertia_ = float(np.sum((X - centroids[labels]) ** 2))
        self.n_iter_ = iteration + 1
        return self                             # return self so fit() is chainable

    def predict(self, X):
        """Assign each point in X to the nearest centroid. Requires prior fit()."""
        if self.cluster_centers_ is None:
            raise RuntimeError("Call fit() before predict().")
        X = np.asarray(X, dtype=float)
        dists = np.linalg.norm(
            X[:, None] - self.cluster_centers_[None, :], axis=2
        )
        return np.argmin(dists, axis=1)

    def fit_predict(self, X):
        """Fit to X and return cluster labels. Convenience wrapper."""
        return self.fit(X).labels_


# ── Quick test ────────────────────────────────────────────────────────────────
km_cls = KMeansScratch(k=4, init='k-means++', random_state=42)
km_cls.fit(X_test2)
print(f"KMeansScratch — inertia: {km_cls.inertia_:.2f}, iterations: {km_cls.n_iter_}")
print(f"Cluster centers shape: {km_cls.cluster_centers_.shape}")


In [ ]:
np.random.seed(42)

# ── Generate synthetic retail dataset (same logic as NB01) ───────────────────
n_customers = 500
segment_sizes = [100, 150, 150, 100]

spend_params = [(2000, 600), (600, 200), (200, 80), (80, 30)]
total_spend = np.concatenate([
    np.random.normal(m, s, n) for (m, s), n in zip(spend_params, segment_sizes)
])
total_spend = np.clip(total_spend, 5, 6000)

freq_params = [(40, 10), (15, 5), (6, 3), (2, 1)]
frequency = np.concatenate([
    np.random.normal(m, s, n) for (m, s), n in zip(freq_params, segment_sizes)
])
frequency = np.clip(frequency, 1, 100).round().astype(int)

recency_params = [(20, 15), (60, 30), (150, 40), (250, 50)]
recency = np.concatenate([
    np.random.normal(m, s, n) for (m, s), n in zip(recency_params, segment_sizes)
])
recency = np.clip(recency, 1, 365).round().astype(int)

shuffle_idx = np.random.permutation(n_customers)
df_retail = pd.DataFrame({
    'CustomerID': np.arange(1001, 1001 + n_customers)[shuffle_idx],
    'TotalSpend':  np.round(total_spend[shuffle_idx], 2),
    'Frequency':   frequency[shuffle_idx],
    'Recency':     recency[shuffle_idx]
})

# ── Apply KMeansScratch to TotalSpend and Frequency ──────────────────────────
X_retail = df_retail[['TotalSpend', 'Frequency']].values

km_retail = KMeansScratch(k=4, init='k-means++', random_state=42)
km_retail.fit(X_retail)
df_retail['Cluster'] = km_retail.labels_

# ── Visualise clusters with annotated centroids ───────────────────────────────
palette = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']
fig, ax = plt.subplots(figsize=(9, 6))

for cid in range(4):
    mask = df_retail['Cluster'] == cid
    ax.scatter(
        df_retail.loc[mask, 'TotalSpend'],
        df_retail.loc[mask, 'Frequency'],
        c=palette[cid], s=50, alpha=0.65,
        edgecolors='white', linewidths=0.4,
        label=f'Cluster {cid} (n={mask.sum()})'
    )

# Plot centroids as black X markers and annotate their coordinates
for cid, (cx, cy) in enumerate(km_retail.cluster_centers_):
    ax.scatter(cx, cy, marker='X', s=250, c='black', zorder=10,
               edgecolors='white', linewidths=0.8)
    ax.annotate(
        f"μ{cid}\n(£{cx:.0f}, {cy:.0f} visits)",
        xy=(cx, cy), xytext=(cx + 60, cy + 1.5),   # offset to avoid overlap
        fontsize=8,
        arrowprops=dict(arrowstyle='->', color='grey', lw=0.8),
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)
    )

ax.set_xlabel("Total Spend (£)", fontsize=12)
ax.set_ylabel("Purchase Frequency (# visits)", fontsize=12)
ax.set_title("KMeansScratch (k=4) Applied to Synthetic Retail Data\nX = centroid position",
             fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
np.random.seed(42)

# ── Verify KMeansScratch against sklearn KMeans ───────────────────────────────
# K-means labels are arbitrary (cluster 0 in scratch may be cluster 2 in sklearn).
# We use the Hungarian algorithm to find the optimal label mapping.

# Fit sklearn KMeans on the same data
sk_km = KMeans(n_clusters=4, init='k-means++', n_init=1,
               random_state=42, max_iter=300)
sk_km.fit(X_retail)
sklearn_labels = sk_km.labels_
scratch_labels  = km_retail.labels_

print(f"Scratch inertia : {km_retail.inertia_:.2f}")
print(f"Sklearn inertia : {sk_km.inertia_:.2f}")
print(f"Difference      : {abs(km_retail.inertia_ - sk_km.inertia_):.2f}")
print()

# Build a contingency matrix: contingency[i, j] = # points that scratch says i, sklearn says j
cont = np.zeros((4, 4), dtype=int)
for s_lbl, sk_lbl in zip(scratch_labels, sklearn_labels):
    cont[s_lbl, sk_lbl] += 1

print("Contingency matrix (rows=scratch labels, cols=sklearn labels):")
print(cont)
print()

# Use the Hungarian algorithm to find the best permutation of labels
# linear_sum_assignment minimises cost; we maximise matches by negating the matrix
row_ind, col_ind = linear_sum_assignment(-cont)   # optimal assignment
matched_count = cont[row_ind, col_ind].sum()      # number of points with matching labels

print(f"Optimal label mapping (scratch -> sklearn): {dict(zip(row_ind.tolist(), col_ind.tolist()))}")
print(f"Points with matching labels after remapping: {matched_count} / {len(scratch_labels)}")
print(f"Agreement: {100 * matched_count / len(scratch_labels):.1f}%")
print()

# Assert high agreement (minor differences may arise from floating-point tie-breaking)
assert matched_count / len(scratch_labels) > 0.95, \
    "Agreement too low — scratch and sklearn diverged significantly!"
print("Assertion passed: scratch and sklearn agree on >95% of assignments.")


## Inertia as Objective: What It Measures and Why We Minimise It

**Inertia** (also called Within-Cluster Sum of Squares, WCSS) is the objective function K-means minimises:

> J = Σ_{i=1}^{n}  ‖x_i − μ_{label(x_i)}‖²

In plain English: for every data point, measure its squared distance to its assigned centroid, then add all those squared distances together.

**Why squared distance?** The mean minimises squared error. Using squared distance ensures the update step (move centroid to cluster mean) is mathematically correct — it provably decreases J.

**The monotone decrease guarantee:**  
- The assignment step can only *decrease or maintain* J (each point moves to a nearer centroid).  
- The update step can only *decrease or maintain* J (the mean is the unique minimiser of within-group squared error).  
- Therefore J is monotonically non-increasing across iterations → convergence guaranteed (finite states).

**Critical flaw:** inertia always decreases as k increases. With k = n (every point is its own cluster), J = 0. This means you **cannot use inertia alone to choose k** — you need the elbow method or silhouette score.


In [ ]:
np.random.seed(42)

# ── Elbow method: plot inertia vs. k for k = 1 to 10 ─────────────────────────
k_range = range(1, 11)                         # try k from 1 to 10
inertias_elbow = []

for k_val in k_range:
    km_tmp = KMeansScratch(k=k_val, init='k-means++', random_state=42)
    km_tmp.fit(X_retail)
    inertias_elbow.append(km_tmp.inertia_)     # record inertia for this k

# ── Plot the elbow curve ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(list(k_range), inertias_elbow,
        marker='o', linewidth=2, markersize=8,
        color='steelblue', markerfacecolor='white', markeredgewidth=2)

# Mark the "elbow" at k=4 (the true number of segments in our data)
elbow_k = 4
ax.axvline(x=elbow_k, color='crimson', linestyle='--', linewidth=1.8,
           label=f'Elbow at k={elbow_k}')

# Annotate inertia values at a few key points
for k_val, inert in zip(k_range, inertias_elbow):
    if k_val in [1, 4, 7, 10]:
        ax.annotate(f'{inert:,.0f}',
                    xy=(k_val, inert), xytext=(k_val + 0.2, inert + 50000),
                    fontsize=8, color='grey')

ax.set_xlabel("Number of Clusters (k)", fontsize=12)
ax.set_ylabel("Inertia (Within-Cluster SS)", fontsize=12)
ax.set_title("Elbow Method — Choosing k for the Retail Dataset", fontsize=13)
ax.legend(fontsize=11)
ax.set_xticks(list(k_range))
plt.tight_layout()
plt.show()

print("The 'elbow' is where inertia stops decreasing as steeply.")
print("Beyond the elbow, adding more clusters gives diminishing returns.")
print(f"For this dataset, k={elbow_k} is the natural choice.")


## K-Means Limitations (Preview)

K-means works well on the blobs we've used so far, but it has structural weaknesses we will explore in Notebook 03:

| Limitation | Why it happens | Solution covered later |
|---|---|---|
| **Assumes spherical clusters** | Uses Euclidean distance to centroids | DBSCAN (NB06), GMM (NB07) |
| **Sensitive to scale** | Features with large ranges dominate distances | StandardScaler (NB03) |
| **Fixed k required** | Must choose k before fitting | Elbow/Silhouette (NB05), DBSCAN |
| **Local optima** | Lloyd's is greedy; K-means++ only partially helps | Multiple random restarts |
| **Outlier sensitivity** | Centroids pulled towards outliers | K-medoids, robust scaling |
| **Slow on very large n** | O(n·k·d) per iteration | Mini-Batch K-Means (NB03) |

Despite these limitations, K-means is often the correct first choice because it is fast, interpretable, and "good enough" for many practical segmentation tasks.


In [ ]:
np.random.seed(42)

# ── Final segmentation: apply KMeansScratch with k=4 to the retail data ──────
# Visualise and attach interpretable segment names.

# Segment profiling: compute mean features per cluster
profile = df_retail.groupby('Cluster')[['TotalSpend', 'Frequency', 'Recency']].mean().round(1)
print("Cluster Profiles (mean values):")
print(profile.to_string())
print()

# Assign business-friendly segment names based on the profiles
# (Sorted by TotalSpend descending — highest spender = Cluster 0 in our setup)
profile_sorted = profile.sort_values('TotalSpend', ascending=False)
segment_names = {}
for rank, (cid, row) in enumerate(profile_sorted.iterrows()):
    if rank == 0:
        segment_names[cid] = 'Champions'
    elif rank == 1:
        segment_names[cid] = 'Loyal Customers'
    elif rank == 2:
        segment_names[cid] = 'At-Risk Customers'
    else:
        segment_names[cid] = 'Lost / One-time'

df_retail['Segment'] = df_retail['Cluster'].map(segment_names)
print("Segment assignments (first 10 rows):")
print(df_retail[['CustomerID', 'TotalSpend', 'Frequency', 'Recency',
                  'Cluster', 'Segment']].head(10).to_string(index=False))

# ── Final visualisation ────────────────────────────────────────────────────────
seg_colours = {
    'Champions': '#4daf4a',
    'Loyal Customers': '#377eb8',
    'At-Risk Customers': '#ff7f00',
    'Lost / One-time': '#e41a1c'
}

fig, ax = plt.subplots(figsize=(10, 6))

for seg_name, colour in seg_colours.items():
    mask = df_retail['Segment'] == seg_name
    ax.scatter(
        df_retail.loc[mask, 'TotalSpend'],
        df_retail.loc[mask, 'Frequency'],
        c=colour, s=55, alpha=0.65,
        edgecolors='white', linewidths=0.4,
        label=f"{seg_name} (n={mask.sum()})"
    )

# Plot centroids
ax.scatter(
    km_retail.cluster_centers_[:, 0],
    km_retail.cluster_centers_[:, 1],
    marker='X', s=280, c='black', zorder=10,
    edgecolors='white', linewidths=0.8, label='Centroids'
)

ax.set_xlabel("Total Spend (£)", fontsize=12)
ax.set_ylabel("Purchase Frequency (# visits)", fontsize=12)
ax.set_title(
    "Final Customer Segmentation — K-Means Scratch (k=4)\n"
    "Business segments labelled by profile",
    fontsize=13
)
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()


## Self-Check Questions

---

**Question 1:**  
Lloyd's algorithm is guaranteed to converge. Does convergence guarantee that we find the **globally optimal** clustering (minimum inertia over all possible assignments)? Explain why or why not.

<details><summary>Show answer</summary>

**No.** Convergence is guaranteed because inertia decreases monotonically and there are only finitely many partitions of n points into k groups — so the process must eventually reach a fixed point (no assignments change).

However, this fixed point is only a **local minimum** of the inertia objective, not necessarily the global minimum. The algorithm can get stuck if the initial centroids happen to be in a bad configuration (all inside one natural cluster, for example).

Solutions: run multiple random restarts and keep the best result (sklearn does this with `n_init=10` by default); use K-means++ to reduce the probability of bad initialisations.

</details>

---

**Question 2:**  
In the K-means++ initialisation, why do we sample the *next* centroid with probability proportional to **D(x)²** rather than just **D(x)**? What would change if we used D(x) instead?

<details><summary>Show answer</summary>

The **D²** (squared distance) weighting is the specific choice that gives the O(log k) approximation guarantee proven by Arthur & Vassilvitskii (2007).

Intuitively, squaring the distance **amplifies the contrast** between nearby and far-away points. If you used D(x) (not squared), a point twice as far from the nearest centroid would be only twice as likely to be chosen. With D², it is four times as likely — much more aggressive in spreading centroids apart.

In practice, using D instead of D² would still be better than uniform random, but would not have the theoretical guarantee and would tend to produce less well-spread initialisations on difficult datasets with clusters of very different sizes.

</details>

---

**Question 3:**  
You run K-means with k=4 on a customer dataset with features `[TotalSpend, Frequency]`. The inertia is 850,000. You then standardise the features (zero mean, unit variance) and re-run. The inertia is now 12.3. Have the clusters changed? Is the second inertia "better"?

<details><summary>Show answer</summary>

The **clusters may have changed** — standardisation changes the relative influence of each feature on the distance calculation. Before standardisation, `TotalSpend` (range £0–£6000) dominates `Frequency` (range 1–100), so K-means effectively ignores frequency. After standardisation, both features contribute equally.

The **two inertia values are not comparable**. Inertia is measured in the same units as the squared features. After standardisation everything is in units of standard deviations, so inertia values are much smaller. You cannot say 12.3 is "better" than 850,000 — they live in different spaces.

**Practical rule:** always standardise features before K-means when they are on different scales. Compare inertia values only across models fitted on the same (same-scale) data.

</details>


In [ ]:
np.random.seed(42)

# ── BONUS: Convergence animation stub ────────────────────────────────────────
# This code creates a matplotlib FuncAnimation that shows Lloyd's iterations.
# Running it in a Jupyter notebook with %matplotlib notebook or in a GUI window
# will display the animation. In nbconvert / static rendering it produces a still.

# Collect all iterations for animation
X_anim, _ = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=5)
anim_snaps = kmeans_steps(X_anim, k=3, random_state=1)
anim_keys = sorted([k for k in anim_snaps.keys() if k != 'init']) + ['final']
# De-duplicate: 'final' might already be the last numeric key
if anim_keys[-2] == anim_keys[-1]:
    anim_keys = anim_keys[:-1]

palette_anim = ['#e41a1c', '#377eb8', '#4daf4a']

fig_anim, ax_anim = plt.subplots(figsize=(6, 5))

def draw_frame(frame_key):
    """Render a single frame of the animation."""
    ax_anim.cla()                               # clear previous frame
    sns.set_theme(style='whitegrid')            # re-apply style after cla()
    labels_f, cents_f = anim_snaps[frame_key]

    for cid in range(3):
        mask = labels_f == cid
        if mask.any():                          # only plot if cluster is non-empty
            ax_anim.scatter(
                X_anim[mask, 0], X_anim[mask, 1],
                c=palette_anim[cid], s=30, alpha=0.6,
                edgecolors='white', linewidths=0.3
            )

    ax_anim.scatter(
        cents_f[:, 0], cents_f[:, 1],
        marker='X', s=180, c='black', zorder=10,
        edgecolors='white', linewidths=0.8
    )
    iter_label = 'Final' if frame_key == 'final' else f'Iteration {frame_key}'
    ax_anim.set_title(f"K-Means Convergence — {iter_label}", fontsize=11)
    ax_anim.set_xlabel("Feature 1")
    ax_anim.set_ylabel("Feature 2")

# Draw the first frame as a static preview (animation requires interactive backend)
draw_frame(1)
plt.tight_layout()
plt.show()

# Build the FuncAnimation object (renders as video/gif if backend supports it)
anim = FuncAnimation(
    fig_anim,
    func=draw_frame,                            # function to call per frame
    frames=anim_keys[:8],                       # show first 8 iterations
    interval=800,                               # ms between frames
    repeat=True
)

print("FuncAnimation object created.")
print("To display in Jupyter: run '%matplotlib notebook' first, then call anim")
print(f"Total frames: {len(anim_keys[:8])}, Interval: 800ms")


## Key Takeaways

| Concept | Summary |
|---|---|
| **Lloyd's algorithm** | Iterate: assign each point to nearest centroid, move centroid to cluster mean. Guaranteed to converge. |
| **Inertia (WCSS)** | Objective being minimised: Σ squared distance from point to its centroid |
| **Local optima** | Convergence is guaranteed but only to a local minimum — run multiple restarts |
| **K-means++ init** | D² sampling spreads initial centroids; lower inertia and variance across runs |
| **Time complexity** | O(n · k · d · i) — linear in sample size, scales well |
| **Elbow method** | Plot inertia vs. k; look for the "elbow" where improvements diminish |
| **Retail application** | 4 segments discovered: Champions, Loyal, At-Risk, Lost/One-time |

---

**Up next — Notebook 03:** K-Means limitations in depth.  
We will see exactly when K-means fails (non-spherical clusters, different densities, outliers) and introduce Mini-Batch K-Means for large-scale data.

> *"All models are wrong, but some are useful. K-means is wrong about shape, scale, and global optimality — and still useful for 80% of segmentation tasks."*
